<font size = "3">

**(Q1)** Make the following changes to the `Fraction` class from Lecture 1.

1. Modify the `__init__` method so that if a user creates a `Fraction` instance with a negative denominator, it returns an equivalent fraction with a positive denominator.

2. Remove the code that uses `gcd` to reduce a `Fraction` to lowest terms from the `__add__` method and add it to the `__init__` method instead. That is, anytime a `Fraction` instance is created, it is automatically put in reduced form.

3. Implement the following remaining simple arithmetic operators to the `Fraction` class: `__sub__`, `__mul__`, and `__truediv__`.

4. Implement the relational operators ``__gt__``, ``__ge__``, ``__lt__``, ``__le__``, and ``__ne__``.


In [3]:
# Q1
def gcd(m, n):
    # Euclid's algorithm
    m, n = abs(m), abs(n)
    while m % n != 0:
        m, n = n, m % n
    return n

class Fraction:
    """Class Fraction"""
    def __init__(self, num, den):
        # (1) if den ==0, raise ValueError
        # and if den <0,  add a negative sign to both the num and den
        if den == 0:
            raise ValueError("Denominator cannot be 0")
        if den < 0:
            num= -num
            den= -den 
        # (2) Reduce it
        common = gcd(num, den)
        self.num = num // common
        self.den = den // common

    def show(self):
        print(f"{self.num}/{self.den}")

    def __str__(self):
        return f"{self.num}/{self.den}"

    def __repr__(self):
        return f"Fraction(num='{self.num}', den='{self.den}')"

    def __add__(self, other_fraction):
        new_num = self.num * other_fraction.den + \
            self.den * other_fraction.num
        new_den = self.den * other_fraction.den 
        return Fraction(new_num, new_den)
    # (3)
    def __sub__(self, other_fraction):
        new_num = self.num * other_fraction.den - \
            self.den * other_fraction.num
        new_den = self.den * other_fraction.den 
        return Fraction(new_num, new_den)
    # a/b * c/d = (ac) / (bd)
    def __mul__(self, other_fraction):
        new_num = self.num * other_fraction.num
        new_den = self.den * other_fraction.den
        return Fraction(new_num, new_den)
    # a/b / c/d = a/b * d/c = (ad) / (bc)
    def __truediv__(self, other_fraction): 
        if other_fraction.num == 0:
            raise ValueError("cannot divide by zero")
        new_num = self.num * other_fraction.den
        new_den = self.den * other_fraction.num
        return Fraction(new_num, new_den)
    # (4)
    # a/b < c/d means a*d<c*b, a/b > c/d means a*d > c*b
    def __lt__(self, other_fraction):
        return self.num * other_fraction.den < other_fraction.num * self.den

    def __le__(self, other_fraction):
        return self.num * other_fraction.den <= other_fraction.num * self.den

    def __gt__(self, other_fraction):
        return self.num * other_fraction.den > other_fraction.num * self.den

    def __ge__(self, other_fraction):
        return self.num * other_fraction.den >= other_fraction.num * self.den

    def __ne__(self, other_fraction):
        return self.num * other_fraction.den != other_fraction.num * self.den


<font size = "3">

**(Q2)** The circuit simulation from lecture 1 works in a backward direction. In other words, given a circuit, the ouptut is produced by working back through the input values, which in turn cause other outputs to be queried. The continues until the external input lines are found, at which point the user is asked for values. Modify the implementation so that the action is in the forward direction; upon receiving inputs the circuit produces an output.

**Hint**: You may develop another way to solve this problem, but I will outline a couple changes I made to develop my solution.

1. I changed the `LogicGate` class to have three attributes: `self.label`, `self.output`, and `self.pin_out`.

2. I changed the `LogicGate` class to have two methods: `send_output(self)` and `set_pin_out(self, target)`

3. I added another attribute to the `Connector` class: `self.value` which holds the output coming out of the "from gate".

4. I wrote a new `Circuit` class that contains a sequence of gates making up the circuit. 

5. After making all of my changes, I executed the circuit with the code below

In [15]:
# 
class LogicGate:
    #(1)
    def __init__(self, lbl):
        self.label = lbl 
        self.output = None
        self.pin_out = None

    def get_output(self):
        self.output = self.perform_gate_logic()   
        return self.output
    #(2) 
    def set_pin_out(self, target):
        self.pin_out = target

    def send_output(self):
        if self.pin_out is None:  
            return  # return if it is the end
        self.pin_out.value = self.output


In [16]:
class BinaryGate(LogicGate):
    def __init__(self, lbl):
        LogicGate.__init__(self, lbl)
        self.pin_a = None
        self.pin_b = None

    def set_pin(self, source):
        if self.pin_a is None:
            self.pin_a = source
        elif self.pin_b is None:
            self.pin_b = source
        else:
            raise RuntimeError("Error: NO AVAILABLE PINS")
    def get_input_a(self):
        # forward: if None -> ask for input
        if self.pin_a is None:
            return int(input(f"Enter pin A input for gate {self.label}:"))
        # else  -> get the value
        else:
            return self.pin_a.value
    def get_input_b(self):
        if self.pin_b is None:
            return int(input(f"Enter pin B input for gate {self.label}:"))
        else:
            return self.pin_b.value        
    
class UnaryGate(LogicGate):
    def __init__(self, lbl):
        LogicGate.__init__(self, lbl)
        self.pin = None

    def set_pin(self, source):
        if self.pin is None:
            self.pin = source
        else:
            raise RuntimeError("Error: NO AVAILABLE PIN")

    def get_input(self):
        if self.pin is None:
            return int(input(f"Enter pin input for gate {self.label}:"))
        else:
            return self.pin.value   

class AndGate(BinaryGate):
    def __init__(self, lbl):
        BinaryGate.__init__(self, lbl)

    def perform_gate_logic(self):
        a = self.get_input_a()
        b = self.get_input_b()
        return a * b


class OrGate(BinaryGate):
    def __init__(self, lbl):
        BinaryGate.__init__(self, lbl)

    def perform_gate_logic(self):
        a = self.get_input_a()
        b = self.get_input_b()
        return max(a, b)


class NotGate(UnaryGate):
    def __init__(self, lbl):
        UnaryGate.__init__(self, lbl)

    def perform_gate_logic(self):
        x = self.get_input()
        return (x + 1) % 2


In [17]:
# (3)
class Connector:
    def __init__(self, from_gate, to_gate):
        self.from_gate = from_gate
        self.to_gate = to_gate
        self.value = None  # adding self.value to Connector

        # 1) connect to the next gate
        to_gate.set_pin(self)
        # 2) connect to the previous gate
        from_gate.set_pin_out(self)


class Circuit:
    def __init__(self, sequence):
        # sequence:[lv1,lv2,lv3]
        self.sequence = sequence

    def run_circuit(self):
        # for gates in each level, compute output and send output
        for level in self.sequence:
            for gate in level:
                gate.get_output()
            for gate in level:
                gate.send_output()

In [18]:
g1 = AndGate("G1")
g2 = AndGate("G2")
g3 = OrGate("G3")
g4 = NotGate("G4")

c1 = Connector(g1, g3)
c2 = Connector(g2, g3)
c3 = Connector(g3, g4)

level_1 = [g1, g2]
level_2 = [g3]
level_3 = [g4]
sequence = [level_1, level_2, level_3]

circuit = Circuit(sequence)

circuit.run_circuit()

<font size = "3">

**(Q3)** Consider the problem of solving an $n\times n$ lower-triangular system of equations $L\mathbf{x} = \mathbf{b}$, where $\mathbf{x}$ is unknown.

$$
\underbrace{\begin{bmatrix}
\ell_{11} & 0         & \cdots & 0 \\
\ell_{21} & \ell_{22} & \ddots & \vdots \\
\vdots    & \vdots    & \ddots & 0 \\
\ell_{n1} & \ell_{n2} & \cdots & \ell_{nn}
\end{bmatrix}}_{L}
\underbrace{\begin{bmatrix}
x_1 \\ x_2 \\ \vdots \\ x_n
\end{bmatrix}}_{\mathbf{x}}
=
\underbrace{\begin{bmatrix}
b_1 \\ b_2 \\ \vdots \\ b_n
\end{bmatrix}}_{\mathbf{b}}.
$$

We assume $\ell_{ii} \neq 0$ for every $i = 1, 2, \dots, n$ so that a unique solution exists.

The *forward-substitution* algorithm can be used to compute the solution as follows:

$$\begin{align*}
x_1 &= \frac{b_1}{\ell_{11}}\\[2pt]
x_2 &= \frac{b_2 - \ell_{21}x_1}{\ell_{22}}\\
x_3 &= \frac{b_3 - \ell_{31}x_1 - \ell_{32}x_2}{\ell_{33}}\\
&\vdots\\
x_i &= \frac{b_i - \sum_{j=1}^{i-1}\ell_{ij}x_j}{\ell_{ii}}\\
&\vdots\\
x_n &= \frac{b_n - \sum_{j=1}^{n-1}\ell_{ij}x_j}{\ell_{nn}}
\end{align*}




1. Implement the forward substitution algorithm as a function `solve_lower_triangular(L, b)`

2. How many operations does your implementation require? "Operations" include assignnents, addition/subtraction, and multiplication/division. Briefly explain how you arrived at your answer.

3. What is the computational cost of the algorithm in terms of "Big-Oh" notation?

4. Verify your answer to part 3 empirically using the `timeit` library for different values of `n`

In [19]:
# L is n×n lower-triangular matrix, b is vector of length n

def solve_lower_triangular(L, b):
    n = len(L)
    # make sure length of L and b are equal
    if len(b) != n:
        raise  ValueError("length of L and b must be equal")
    
    x = [0] * n # x is vector of length n

    for i in range(n):
        diag = L[i][i] # to compute xi, get the denominator Lij in the equation
        # Compute sum_{j=0}^{i-1} L[i][j] * x[j]
        sum = 0
        for j in range(i):
            sum += L[i][j] * x[j]

        x[i] = (b[i] - sum) / diag

    return x

In [1]:
import numpy as np 

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Here's a good way to create lower triangular systems with unique solutions


n = 50 

L = np.tril(np.random.randn(n,n))
L += np.diag(5*(np.random.rand(n,) + 1)) # ensure matrix is "sufficiently" non-singular

b = np.random.randn(n,)

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Here's how you can test if your function was implemented correctly

def solve_triangular(L, b):
    # will obviously need to change this
    return np.linalg.solve(L, b)

n = 50

L = np.tril(np.random.randn(n,n))
L += np.diag(5*(np.random.rand(n,) + 1)) # ensure matrix is "sufficiently" non-singular

x_true = np.ones(n,)
b = L @ x_true 

my_x = solve_triangular(L, b)

# true solution should be all ones... should get a really small number here
print(max(abs(my_x - 1)))

<font size = "4">

**(Q4)** Consider the function $f(n) = \frac{2}{3}n^2 + 500n + 50$. 

- Using the precise definition of $\mathcal{O}(\cdot)$, show that $f(n) = \mathcal{O}(n^2)$

- Using the precise definition of $\mathcal{\Theta}(\cdot)$, show that $f(n) = \mathcal{\Theta}(n^2)$

<font size = "3">

**(Q5)** Implement a class `TransformHistory` that contains a Pandas `DataFrame` object along with an "undo_stack" and a "redo_stack". This class allows a user to apply a sequence of transformations to a DataFrame while providing a mechanism for undoing/redoing transformations. The outline of the class is below (using the `Stack` class from Lecture 4).

**Hint:** Think of the "Back" and "Forward" options in your browser, or the "undo" "redo" features when writing a word document.

In [ ]:
class TransformHistory:
    def __init__(self, df):
        self.data = df.copy()
        self.undo_stack = Stack()
        self.redo_stack = Stack()

    def set_data(self, df):
        """Set the working DataFrame
        - overwrite self.data with a copy of df
        - reset both undo and redo stacks"""
        pass

    def apply(self, fn):
        """ Apply the transformation fn to the DataFrame
        - Clear the redo stack
        - Add a copy of the current data to the undo stack
        - Overwrite the current data by applying the 
            transformation to a copy of the current data"""

        pass

    def undo(self):
        """Undo the most recent transformation.
        If there is no transformation to undo, leave
        data unchanged"""
        pass

    def redo(self):
        """Redo the most recent 'undone' transformation.
        If there is no transformation to redo, leave
        data unchanged""" 
        pass


<font size = "3">

See below for example usage of the `TransformHistory` class.

In [ ]:
import pandas as pd

def add_y(d):
    return d.assign(y=d["x"] + 10)

def filter_ge_2(d):
    return d.query("x >= 2")


# initialize with empty DataFrame
h = TransformHistory(pd.DataFrame())

# replace with simple 1-column data frame
h.set_data(pd.DataFrame({"x": [1, 2, 3]}))

h.apply(add_y)
print(h.data, "\n")
#     x   y
# 0   1  11
# 1   2  12
# 2   3  13

h.apply(filter_ge_2, "filter")
print(h.data, "\n")
#     x   y
# 1   2  12
# 2   3  13

h.undo()
print(h.data, "\n")
#     x   y
# 0   1  11
# 1   2  12
# 2   3  13

h.redo()
print(h.data, "\n")
#     x   y
# 1   2  12
# 2   3  13